You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Title: The Future of Artificial Intelligence: Trends, Players, and News

Introduction:
Artificial Intelligence (AI) is rapidly transforming industries and shaping the future of technology. In this blog article, we will delve into the latest trends, key players, and noteworthy news in the field of AI to provide our audience with valuable insights and information.

Key Points:
1. Latest Trends in Artificial Intelligence:
- Machine learning and deep learning advancements
- AI-powered automation in various industr

I now can give a great answer

Final Answer:
# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction

Artificial Intelligence (AI) is revolutionizing various industries and driving technological advancements at a rapid pace. In this blog article, we will explore the latest trends, key players, and noteworthy news in the field of AI to provide our audience with valuable insights and updates.

## Latest Trends in Artificial Intelligence

One of the most significant trends in AI is the continuous advancement in machine learning and deep learning technologies. These innovations have enabled AI-powered automation in industries such as manufacturing, finance, and customer service, leading to increased efficiency and productivity. Moreover, ethical considerations in AI development have gained prominence, with a focus on ensuring transparency and fairness in AI algorithms.

The integration of AI with the Internet of Things (IoT) is another trend that is shaping the 

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction

Artificial Intelligence (AI) is revolutionizing various industries and driving technological advancements at a rapid pace. In this blog article, we will explore the latest trends, key players, and noteworthy news in the field of AI to provide our audience with valuable insights and updates.

## Latest Trends in Artificial Intelligence

One of the most significant trends in AI is the continuous advancement in machine learning and deep learning technologies. These innovations have enabled AI-powered automation in industries such as manufacturing, finance, and customer service, leading to increased efficiency and productivity. Moreover, ethical considerations in AI development have gained prominence, with a focus on ensuring transparency and fairness in AI algorithms.

The integration of AI with the Internet of Things (IoT) is another trend that is shaping the future of technology. AI-powered IoT devices are enabling smart homes, autonomous vehicles, and predictive maintenance in various sectors. Additionally, AI is making significant strides in healthcare and personalized medicine, with applications ranging from disease diagnosis to treatment customization.

## Key Players in Artificial Intelligence

Several key players are driving the development and adoption of AI technologies. Google's DeepMind has made significant breakthroughs in AI research, particularly in areas such as reinforcement learning and natural language processing. IBM Watson is a leader in enterprise AI solutions, offering cognitive computing capabilities for businesses across industries.

Microsoft Azure AI and Amazon Web Services AI are leading cloud-based AI platforms that provide scalable and efficient AI solutions for organizations. OpenAI, founded by Elon Musk and Sam Altman, is dedicated to advancing friendly AI for the benefit of all humanity.

## Noteworthy News in Artificial Intelligence

Recent developments in natural language processing have led to breakthroughs in AI applications such as chatbots, virtual assistants, and language translation tools. AI is also playing a crucial role in the advancement of autonomous vehicles, with companies like Tesla and Waymo making significant strides in self-driving technology.

In the e-commerce sector, AI-driven personalized recommendations are enhancing the shopping experience for consumers, increasing engagement and conversions. In cybersecurity, AI is being utilized for threat detection and prevention, helping organizations defend against cyber attacks. Additionally, updates on AI ethics and regulations are shaping the responsible development and deployment of AI technologies.

In conclusion, the future of Artificial Intelligence is promising, with continuous advancements in technology, ethical considerations, and industry applications. By staying informed about the latest trends, key players, and news in AI, our audience can make informed decisions and stay ahead in this ever-evolving field.

## Call to Action

Stay updated on the future of AI by subscribing to our newsletter for regular insights and updates on Artificial Intelligence trends, key players, and news.

By incorporating the latest trends, key players, and noteworthy news in Artificial Intelligence, this blog article aims to provide our audience with valuable and engaging content that will help them navigate the complexities of AI and make informed decisions in their respective industries.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on YOUR TOPIC HERE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer: 

Title: Unveiling the Latest Trends in Sustainable Fashion

Introduction:
- Brief overview of the importance of sustainable fashion
- Mention the growing interest in eco-friendly and ethically made clothing
- Highlight the impact of fast fashion on the environment and society

Key Points:
1. Latest Trends in Sustainable Fashion
- Rise of eco-friendly fabrics such as organic cotton, Tencel, and bamboo
- Increasing popularity of upcycling and zero-waste design
- Emphasis on transparency and ethical production practices

I now can give a great answer

Final Answer: 

# Unveiling the Latest Trends in Sustainable Fashion

In today's fast-paced fashion industry, the importance of sustainable practices has become more prevalent than ever. With a growing interest in eco-friendly and ethically made clothing, consumers are seeking alternatives to the harmful impacts of fast fashion on the environment and society. As a result, the latest trends in sustainable fashion are shaping the industry towards a more conscious and responsible approach.

## Latest Trends in Sustainable Fashion
One of the key trends in sustainable fashion is the rise of eco-friendly fabrics such as organic cotton, Tencel, and bamboo. These materials not only reduce the environmental footprint of clothing production but also offer a higher level of comfort and quality for consumers. Additionally, there is an increasing popularity of upcycling and zero-waste design, where designers reimagine and repurpose existing materials to create new and

In [14]:
Markdown(result)

# Unveiling the Latest Trends in Sustainable Fashion

In today's fast-paced fashion industry, the importance of sustainable practices has become more prevalent than ever. With a growing interest in eco-friendly and ethically made clothing, consumers are seeking alternatives to the harmful impacts of fast fashion on the environment and society. As a result, the latest trends in sustainable fashion are shaping the industry towards a more conscious and responsible approach.

## Latest Trends in Sustainable Fashion
One of the key trends in sustainable fashion is the rise of eco-friendly fabrics such as organic cotton, Tencel, and bamboo. These materials not only reduce the environmental footprint of clothing production but also offer a higher level of comfort and quality for consumers. Additionally, there is an increasing popularity of upcycling and zero-waste design, where designers reimagine and repurpose existing materials to create new and unique pieces. This shift towards circular fashion practices promotes resource efficiency and minimizes waste in the industry. Moreover, there is a growing emphasis on transparency and ethical production practices, with brands showcasing their supply chain processes and commitments to fair labor standards.

## Key Players in the Sustainable Fashion Industry
Leading sustainable fashion brands like Patagonia, Stella McCartney, and Reformation have been at the forefront of promoting ethical and eco-friendly practices in the industry. These brands prioritize sustainability in every aspect of their operations, from sourcing materials to manufacturing processes. Additionally, influencers and celebrities play a significant role in promoting sustainable fashion by endorsing and wearing ethical clothing brands. Their influence helps raise awareness and drive consumer demand for sustainable products. Major retailers have also started implementing sustainable fashion initiatives to meet the changing preferences of conscious consumers and reduce their environmental impact.

## Noteworthy News in Sustainable Fashion
Recent collaborations between sustainable brands and mainstream retailers have brought sustainable fashion to a wider audience, making eco-friendly options more accessible. Moreover, there have been updates on legislation and regulations promoting ethical fashion practices, encouraging more brands to adopt sustainable measures. Success stories of sustainable fashion startups and initiatives showcase the growing potential and impact of sustainable practices in the industry. These developments highlight the positive changes happening in the fashion world towards a more sustainable and ethical future.

# Call to Action
As consumers, we have the power to support sustainable fashion by choosing to shop from ethical brands that prioritize eco-friendly and ethical practices. By making informed purchasing decisions, we can contribute to the shift towards a more sustainable industry. Additionally, incorporating sustainable practices into our own wardrobe, such as buying second-hand clothing, repairing and reusing garments, and recycling old textiles, can make a significant difference in reducing our environmental impact. Advocating for sustainability in the fashion industry by supporting initiatives, signing petitions, and raising awareness can also drive positive change and encourage more brands to embrace ethical and eco-friendly practices.

In conclusion, the latest trends in sustainable fashion are reshaping the industry towards a more conscious and responsible future. By staying informed and making thoughtful choices, we can support the growth of sustainable practices and contribute to a more sustainable fashion industry for generations to come. Let's embrace sustainable fashion and make a positive impact on the world around us.

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).